In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )

In [ ]:
# System parameters.
μ = 1.0
N = defInt( 100 )
L = √(N/μ)

# Case and data folder.
ξ = 0.1
case = "../data/results/case-0/"
folder = case*"N-$(N)/mu-$(round( μ, digits=3 ))_xi-$(round( ξ, digits=5 ))/"

# Import parameters from simulated data.
scale = loadscale( folder*"sim-scale.json" )
params = loadparams( folder*"sim-params.json" )
nondim = Nondim( params; scale=scale )
println( "   dim. parameters: ", params )
println( "nondim. parameters: ", nondim )

# Time-step length.
δt = Δt;

In [ ]:
# Run simulation under each environment parameter.
T = 1000;  Tload = 500/scale.T;  M = 100
Nt = round( defInt, T/δt );  tlist = 1:Nt
nt = round( defInt, 1/(2*δt*scale.T) );  tsave = Set( 1:nt:Nt );

# Frequency of adjacency calculation.
δt̂ = round( defInt, 0.1/δt )
println( "δt̂ = $(δt̂)" )

In [ ]:
# Initialize data variables.
xdata = [Matrix{defFloat}( undef, length( tsave ) + 1, 3 ) for _ ∈ 1:M]
zdata = Matrix{State}( undef, M, length( tsave ) + 1 )

# Run simulation.
@threads for m ∈ 1:M
    # If steps are already saved, use as initial state.
    file = folder*"steps/state_T-$(Tload)_m-$(m).txt"

    # Initialize agent states.
    z = initialstate( N, L/scale.L; A=1, file=file )
    ẑ = copystate( z )

    # Initialize adjacency and saved state.
    A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ )

    # Initialize saved states.
    zdata[m,1] = copystate( ẑ )
    xdata[m][1,:] = statecomposition( N, ẑ )

    # Run simulation.
    t̂ = 2
    for t ∈ tlist[1:end]
        # Update the adjacency matrix.
        if (t % δt̂) == 0
            A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ )
        end

        # Step simulation.
        step!( N, L, nondim, z, ẑ; δt=δt, A=A )

        # Save state if in appropriate subset.
        if t ∈ tsave
            zdata[m,t̂] = copystate( ẑ )
            xdata[m][t̂,:] = statecomposition( N, ẑ )
            t̂ += 1
        end

        # Swap contents.
        tmp = z;  z = ẑ;  ẑ = tmp
    end
end

In [ ]:
# Plot the mean activity deries for each duration value.
plt = plot( size=(300,200), xformatter=:plain, margin=2mm, dpi=600 )

for m ∈ 1:M
    alist = xdata[m][:,1]
    if m == M
        plot!( plt, δt*(0:nt:Nt), alist; color=:cornflowerblue, alpha=1, lw=2, label="" )
    else
        plot!( plt, δt*(0:nt:Nt), alist; color=:black, alpha=1/6, label="" )
    end
end

plot!( plt; xlims=(0,T), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1), ylabel="proportion of\nants active, "*L"a" )

In [ ]:
if false
    if !isdir( folder*"sim/" )
        mkpath( folder*"sim/" )
    end

    for m ∈ 1:M
        xdata = [z.x for z ∈ zdata[m,:]]
        ydata = [z.y for z ∈ zdata[m,:]]
        θdata = [z.θ for z ∈ zdata[m,:]]

        writedlm( folder*"sim/x-state_m-$(m).txt", xdata )
        writedlm( folder*"sim/y-state_m-$(m).txt", ydata )
        writedlm( folder*"sim/theta-state_m-$(m).txt", θdata )
    end
end